In [1]:
from langchain_openai import ChatOpenAI
from langchain_google_vertexai import ChatVertexAI

from wsd.load_data import load_data
from wsd.models import BinaryWSD, ClusterByMeaningModel, DummyComparator
from linpub.metrics import accuracy

In [ ]:
X, y = load_data()

k = 200
X_test, y_test = X[:k], y[:k]

gpt = ChatOpenAI(temperature=0, model="gpt-4o-mini").with_structured_output(BinaryWSD)
model1 = ClusterByMeaningModel(comparator=gpt)

gemini = ChatVertexAI(temperature=0, model="gemini-1.5-flash").with_structured_output(BinaryWSD)
model2 = ClusterByMeaningModel(comparator=gemini)

dummy = DummyComparator()
model3 = ClusterByMeaningModel(comparator=dummy)

y_pred1 = model1.predict(X_test, verbose=True)
y_pred2 = model2.predict(X_test, verbose=True)
y_pred3 = model3.predict(X_test, verbose=True)

print(f"gpt-4o: accuracy  = {accuracy(y_pred1, y_test)}")
print(f"gemini1.5-flash: accuracy  = {accuracy(y_pred2, y_test)}")
print(f"dummy: accuracy  = {accuracy(y_pred3, y_test)}")

  3%|▎         | 3/98 [00:06<04:00,  2.53s/it]

In [ ]:
import pandas as pd

records = []
for yp, yt, c in zip(y_pred3, y_test, X_test):
    r = {'lemma': c.lemma, 'pos': c.pos, 'y': yt, 'y_pred': yp, 'text': c.text, 'context': c.context}
    records.append(r)
pd.DataFrame(records).sort_values(['lemma', 'pos']).to_csv('plop.csv', index=False)